In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
seed=2017

df=pd.read_csv('diabetes.csv')

X=df.iloc[:,:8].values
y=df['Outcome'].values

X=StandardScaler().fit_transform(X)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=seed,shuffle=True)

kfold=StratifiedKFold(n_splits=5,random_state=seed,shuffle=True)
num_trees=100
clf_rf=RandomForestClassifier(random_state=seed).fit(X_train,y_train)

rf_params={
    'n_estimators':[100,250,500,750,1000],
    'criterion':['gini','entropy'],
    'max_features':['sqrt','log2'],
    'max_depth':[1,3,5,7,9]
}

grid=GridSearchCV(clf_rf,rf_params,scoring='roc_auc',cv=kfold,verbose=10,n_jobs=-1)
grid.fit(X_train,y_train)

print('Best Parameters:',grid.best_params_)

results=cross_val_score(grid.best_estimator_,X_train,y_train,cv=kfold)

print("Accuracy Train CV:",results.mean())
print("Accuracy Train:",metrics.accuracy_score(grid.best_estimator_.predict(X_train),y_train))
print("Accuracy Test:",metrics.accuracy_score(grid.best_estimator_.predict(X_test),y_test))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters: {'criterion': 'entropy', 'max_depth': 5, 'max_features': 'log2', 'n_estimators': 500}
Accuracy Train CV: 0.7522499134648667
Accuracy Train: 0.8621973929236499
Accuracy Test: 0.7965367965367965


In [17]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint as sp_randint

param_dist={'n_estimators':sp_randint(100,1000),
            'criterion':['gini','entropy'],
            'max_features':['sqrt','log2'],
            'max_depth':[None,1,3,5,7,9]
            }

n_iter_search=20

random_search=RandomizedSearchCV(clf_rf,param_distributions=param_dist,
                                 cv=kfold,n_iter=n_iter_search,verbose=10,
                                 n_jobs=-1,random_state=seed)
random_search.fit(X_train,y_train)

print('Best Parameters:',random_search.best_params_)

results=cross_val_score(random_search.best_estimator_,
                        X_train,y_train,cv=kfold)

print("Accuracy Train CV:",results.mean())
print("Accuracy Train:",metrics.accuracy_score(random_search.predict(X_train),y_train))
print("Accuracy Test:",metrics.accuracy_score(random_search.predict(X_test),y_test))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'criterion': 'entropy', 'max_depth': 5, 'max_features': 'sqrt', 'n_estimators': 853}
Accuracy Train CV: 0.7577881619937694
Accuracy Train: 0.845437616387337
Accuracy Test: 0.7878787878787878
